In [1]:
# script to save the rankings for the mechanism
import os
import re
import sys
import glob
import copy
import logging
import yaml
import pickle
import subprocess
import numpy as np
import pandas as pd

import rmgpy.data.kinetics
import rmgpy.chemkin
import cantera as ct

import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
sys.path.append(os.environ['DATABASE_DIR'])
import database_fun

Loading DFT database from /projects/westgroup/harris.se/autoscience/reaction_calculator/database


In [3]:
working_dir = '/scratch/harris.se/guassian_scratch/test_sens_20260313'


In [4]:
input_chemkin = os.path.join(working_dir, 'chem_annotated.inp')
dictionary = os.path.join(working_dir, 'species_dictionary.txt')
cantera_file = os.path.join(working_dir, 'chem_annotated.yaml')
analysis_dir = os.path.join(working_dir, 'analysis')
os.makedirs(analysis_dir, exist_ok=True)

species_list, reaction_list = rmgpy.chemkin.load_chemkin_file(input_chemkin, dictionary_path=dictionary, use_chemkin_names=True)

gas = ct.Solution(cantera_file)

# This cti -> rmg converter dictionary can be made using rmg_tools/ct2rmg_dict.py
RMG_TOOLS_DIR = '/home/harris.se/rmg/rmg_tools'
if not os.path.exists(os.path.join(working_dir, 'ct2rmg_rxn.pickle')):
    print('Creating ct2rmg pickle')
    subprocess.run(['python', os.path.join(RMG_TOOLS_DIR, 'ct2rmg_dict.py'), input_chemkin])

with open(os.path.join(working_dir, 'ct2rmg_rxn.pickle'), 'rb') as handle:
    ct2rmg_rxn = pickle.load(handle)
    


print(f'{len(species_list)} species loaded')
print(f'{len(reaction_list)} reactions loaded')

71 species loaded
371 reactions loaded


In [5]:
# load base and perturbed delays and check size
base_delays = np.load(os.path.join(working_dir, 'base_delays.npy'))
base_delays = np.array(np.repeat(np.matrix(base_delays), gas.n_species + gas.n_reactions, axis=0))
total_delays = np.load(os.path.join(working_dir, 'total_perturbed_mech_delays.npy'))

# check the size
conditions_dict_path = os.path.join(working_dir, 'sim_config.yaml')
if not os.path.exists(conditions_dict_path):
    logging.warning(f'Expected to find sim_config.yaml at {conditions_dict_path} but it does not exist. Please copy it to the directory with your mech file.')
    raise FileNotFoundError(f'sim_config.yaml not found at {conditions_dict_path}')

with open(conditions_dict_path) as f:
    conditions_dict = yaml.safe_load(f)
K = len(conditions_dict['sensitivity_points'])

assert total_delays.shape[0] == gas.n_species + gas.n_reactions
assert total_delays.shape[1] == K

In [6]:
N = len(gas.species())
M = len(gas.reactions())

In [8]:
rxn_uncertainty_file = os.path.join(working_dir, 'gao_reaction_uncertainty.npy')
sp_uncertainty_file = os.path.join(working_dir, 'gao_species_uncertainty.npy')

rmg_rxn_uncertainty = np.load(rxn_uncertainty_file)
rmg_sp_uncertainty = np.load(sp_uncertainty_file)

assert len(rmg_rxn_uncertainty) == len(reaction_list)
assert len(rmg_sp_uncertainty) == len(species_list)


rxn_uncertainty = np.zeros(gas.n_reactions)
for ct_index in range(len(rxn_uncertainty)):
    rxn_uncertainty[ct_index] = rmg_rxn_uncertainty[ct2rmg_rxn[ct_index]]

# Cantera species should be in same rmg order, but this makes sure for us
for i in range(len(species_list)):
    if str(species_list[i]) != gas.species_names[i]:
        print(i)
    assert str(species_list[i]) == gas.species_names[i]

sp_uncertainty = rmg_sp_uncertainty

total_uncertainty_array = np.concatenate((sp_uncertainty, rxn_uncertainty), axis=0)
total_uncertainty_mat = np.array(np.repeat(np.transpose(np.matrix(total_uncertainty_array)), K, axis=1))


## Gather Uncertainty Rankings

In [9]:
SPECIES_DFT_ERROR = 1.5
REACTION_DFT_ERROR = 1 / np.sqrt(3) * np.log(10)

sp_dft_uncertainty_mat = np.ones((N, K)) * SPECIES_DFT_ERROR
rxn_dft_uncertainty_mat = np.ones((M, K)) * REACTION_DFT_ERROR
dft_uncertainty_mat = np.concatenate((sp_dft_uncertainty_mat, rxn_dft_uncertainty_mat), axis=0)


reaction_indices = np.arange(0, len(gas.reactions()))
reaction_uncertainty_order = [x for _,x in sorted(zip(rxn_uncertainty, reaction_indices))][::-1]

In [10]:
1 / np.sqrt(3) * np.log(10)

1.3293981232721321

In [11]:
print('Top Uncertain Reactions')
print('i\tDelta\tReaction\tSensitivity\tImprovement Score')
for i in range(0, 10):
    ct_index = reaction_uncertainty_order[i]
    print(ct_index, '\t', np.round(rxn_uncertainty[ct_index], 3),
          '\t', gas.reactions()[ct_index], 
          '\t', reaction_list[ct2rmg_rxn[ct_index]].family)

# TODO convert to db indices? 

Top Uncertain Reactions
i	Delta	Reaction	Sensitivity	Improvement Score
360 	 22.863 	 CC[O](72) + O2(2) <=> C2H4O(74) + HO2(16) 	 Disproportionation
313 	 15.147 	 C4H9O4(154) + H(14) <=> CC(CCOO)OO(178) 	 R_Recombination
312 	 15.147 	 C4H9O4(138) + H(14) <=> CC(CCOO)OO(178) 	 R_Recombination
311 	 15.147 	 C4H9O4(155) + H(14) <=> CC(CCOO)OO(178) 	 R_Recombination
309 	 15.147 	 C[CH]CCOO(257) + HO2(16) <=> CC(CCOO)OO(178) 	 R_Recombination
199 	 15.147 	 O2(2) + [CH2]CCCOO(258) <=> [O]OCCCCOO(272) 	 R_Recombination
194 	 15.147 	 C[CH]CCOO(257) + O2(2) <=> C4H9O4(155) 	 R_Recombination
148 	 15.147 	 C[CH]C(C)OO(92) + O2(2) <=> C4H9O4(137) 	 R_Recombination
144 	 15.147 	 O2(2) + [CH2]CC(C)OO(94) <=> C4H9O4(138) 	 R_Recombination
335 	 14.71 	 C4H9O4(154) + C[CH]CC(23) <=> C4H8(47) + CC(CCOO)OO(178) 	 Disproportionation


## Gather Sensitivity Rankings

In [12]:
assert total_delays.shape == base_delays.shape
total_delays[total_delays == 0] = np.nan


In [13]:
total_delays.shape

(451, 6)

In [14]:
d_ln_tau = np.log(total_delays) - np.log(base_delays)
avg_d_ln_tau = np.nanmean(d_ln_tau, axis = 1)
avg_d_ln_tau[np.isnan(avg_d_ln_tau)] = -np.inf

### Get $\Delta G$ for each parameter

In [15]:
delta_G_kcal_mol = np.zeros((N, K)) + 0.1

### Get $\Delta \ln k$ for each parameter

In [16]:
# except we know that by definition, this is 0.1
delta_ln_k = 0.1 * np.ones((M, K))

In [17]:
# concatenate into a big delta matrix
delta = np.concatenate((delta_G_kcal_mol, delta_ln_k), axis=0)

### Put it all together into $\frac{\partial \ln \tau}{\partial G}$ or $\frac{\partial \ln \tau}{\partial \ln k}$

In [18]:
# first derivative is change in delay / change in G
first_derivative = np.divide(d_ln_tau, delta)

In [19]:
first_derivative.shape

(451, 6)

### Display most sensitive parameters

In [20]:
avg_first_derivative = np.nanmean(first_derivative, axis=1)

abs_avg_first_derivative = np.abs(avg_first_derivative)
abs_avg_first_derivative[np.isnan(abs_avg_first_derivative)] = -np.inf


parameter_indices = np.arange(0, N + M)
reaction_sensitivity_order = [x for _, x in sorted(zip(abs_avg_first_derivative, parameter_indices))][::-1]

print('Top Sensitive Parameters')
print('i\tct idx\tSensitivity\tParameter')
for i in range(0, 20):
    ct_index = reaction_sensitivity_order[i]
    if ct_index < N:
        print(i, '\t', ct_index, '\t', np.round(abs_avg_first_derivative[ct_index], 9),
              '\t', gas.species()[ct_index], )
    else:
        print(i, '\t', ct_index, '\t', np.round(abs_avg_first_derivative[ct_index], 9),
              '\t', gas.reactions()[ct_index - N])

Top Sensitive Parameters
i	ct idx	Sensitivity	Parameter
0 	 141 	 0.435889116 	 OH(15) + butane(1) <=> H2O(8) + [CH2]CCC(24)
1 	 5 	 0.433223834 	 <Species O2(2)>
2 	 140 	 0.424261762 	 OH(15) + butane(1) <=> C[CH]CC(23) + H2O(8)
3 	 19 	 0.418431172 	 <Species HO2(16)>
4 	 224 	 0.301884297 	 C4H9O4(138) <=> C4H8O3(153) + OH(15)
5 	 142 	 0.282991073 	 C[CH]CC(23) + H2O2(17) <=> HO2(16) + butane(1)
6 	 4 	 0.250981581 	 <Species butane(1)>
7 	 121 	 0.246999334 	 [CH3](22) + butane(1) <=> CH4(10) + C[CH]CC(23)
8 	 152 	 0.246194435 	 [CH2]CCC(24) <=> C2H4(11) + C[CH2](20)
9 	 92 	 0.235887695 	 H2O2(17) (+M) <=> 2 OH(15) (+M)
10 	 20 	 0.225015795 	 <Species H2O2(17)>
11 	 48 	 0.196231846 	 <Species C4H9O4(138)>
12 	 42 	 0.166464858 	 <Species [CH2]CC(C)OO(94)>
13 	 24 	 0.143829417 	 <Species [CH3](22)>
14 	 210 	 0.140776368 	 C4H8(48) + HO2(16) <=> CCC(C)O[O](60)
15 	 166 	 0.134052461 	 C[CH]CC(23) + O2(2) <=> C4H8(47) + HO2(16)
16 	 215 	 0.131040188 	 O2(2) + [CH2]CC(C)OO(94)

# Compute Improvement Score

## Notes
- Don't average the improvement scores until the very end. you're confounding different reactor settings and it doesn't make sense

In [21]:
delta_uncertainty_squared = np.float_power(total_uncertainty_mat, 2.0) - np.float_power(dft_uncertainty_mat, 2.0)
sensitivity_squared = np.float_power(first_derivative, 2.0)

improvement_score = np.multiply(delta_uncertainty_squared, sensitivity_squared)

avg_improvement_score = np.nanmean(improvement_score, axis=1)
avg_improvement_score[np.isnan(avg_improvement_score)] = -np.inf

improvement_score[np.isnan(improvement_score)] = -np.inf


total_uncertainty_squared = np.nansum(np.multiply(sensitivity_squared, np.float_power(total_uncertainty_mat, 2.0)), axis=0)
total_uncertainty = np.array(np.float_power(total_uncertainty_squared, 0.5)).ravel()


In [22]:
# # Save the matrices for convenience
np.save(os.path.join(analysis_dir, 'total_uncertainty_mat'), total_uncertainty_mat)
np.save(os.path.join(analysis_dir, 'dft_uncertainty_mat'), dft_uncertainty_mat)
np.save(os.path.join(analysis_dir, 'first_derivative'), first_derivative)
np.save(os.path.join(analysis_dir, 'improvement_score'), improvement_score)


In [23]:
# # load from files if you want to skip everything
# total_uncertainty_mat2 = np.load(os.path.join(analysis_dir, 'total_uncertainty_mat.npy'))
# dft_uncertainty_mat2 = np.load(os.path.join(analysis_dir, 'dft_uncertainty_mat.npy'))
# first_derivative2 = np.load(os.path.join(analysis_dir, 'first_derivative.npy'))
# improvement_score2 = np.load(os.path.join(analysis_dir, 'improvement_score.npy'))

### Display Top Improvement Scores

In [24]:
parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


# compute improvement total - sum of all possible improvements to make
improvement_total = np.sum(avg_improvement_score[avg_improvement_score > 0])


print('Top Improvement Scores')
print('i\tCt Index\tDb Index\tImprovement Score\tImprovement %\tReaction')
for i in range(0, 200):
    ct_index = improvement_order[i]
    
    
    if ct_index < N:
        db_index = database_fun.get_unique_species_index(species_list[ct_index])
        
        print(i, '\t', ct_index, '\t\t', db_index, '\t', np.round(avg_improvement_score[ct_index], 9),
              '\t', gas.species()[ct_index], )
    else:
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[ct_index - N]].family
        except AttributeError:
            pass
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
        print(i, '\t', ct_index - N, '\t\t', db_index, '\t', np.round(avg_improvement_score[ct_index], 9),
              '\t', np.round(avg_improvement_score[ct_index] / improvement_total, 9), '\t', gas.reactions()[ct_index - N], family)

Top Improvement Scores
i	Ct Index	Db Index	Improvement Score	Improvement %	Reaction
0 	 144 		 281 	 10.596339781 	 0.463610525 	 O2(2) + [CH2]CC(C)OO(94) <=> C4H9O4(138) R_Recombination
1 	 70 		 288 	 3.10131357 	 0.135688515 	 OH(15) + butane(1) <=> H2O(8) + [CH2]CCC(24) H_Abstraction
2 	 92 		 313 	 1.923164286 	 0.084142187 	 O2(2) + [CH2]CCC(24) <=> C4H8(48) + HO2(16) Disproportionation
3 	 71 		 4724 	 1.714669645 	 0.07502014 	 C[CH]CC(23) + H2O2(17) <=> HO2(16) + butane(1) H_Abstraction
4 	 95 		 278 	 1.454103227 	 0.063619851 	 C[CH]CC(23) + O2(2) <=> C4H8(47) + HO2(16) Disproportionation
5 	 50 		 245 	 0.750106284 	 0.032818612 	 [CH3](22) + butane(1) <=> CH4(10) + C[CH]CC(23) H_Abstraction
6 	 24 		 21 	 0.479395161 	 <Species [CH3](22)>
7 	 81 		 293 	 0.446134471 	 0.019519253 	 [CH2]CCC(24) <=> C2H4(11) + C[CH2](20) PDEP
8 	 48 		 74 	 0.334838104 	 <Species C4H9O4(138)>
9 	 72 		 4736 	 0.243149341 	 0.010638258 	 H2O2(17) + [CH2]CCC(24) <=> HO2(16) + butane(1) H_Abst

In [25]:
database = rmgpy.data.rmg.RMGDatabase()

database.load(
    path = rmgpy.settings['database.directory'],
    thermo_libraries = ['BurkeH2O2', 'primaryThermoLibrary'],
    transport_libraries = [],
    reaction_libraries = [],
    seed_mechanisms = [],
    kinetics_families = ['Disproportionation', 'H_Abstraction', 'R_Addition_MultipleBond', 'intra_H_migration'],
    kinetics_depositories = ['training'],
    #frequenciesLibraries = self.statmechLibraries,
    depository = False,
)
for family in database.kinetics.families:
    if not database.kinetics.families[family].auto_generated:
        database.kinetics.families[family].add_rules_from_training(thermo_database=database.thermo)
        database.kinetics.families[family].fill_rules_by_averaging_up(verbose=True)


ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.8846,97.4337,105.393],'J/(mol*K)'), H298=(-477.191,'kJ/mol'), S298=(269.551,'J/(mol*K)'), Cp0=(33.2579,'J/(mol*K)'), CpInf=(103.931,'J/(mol*K)'), comment="""Thermo group additivity estimation: group(O2s-(Cds-Cd)(Cds-Cd)) + group(O2s-(Cds-O2d)H) + group(Cds-OdOsOs) + group(Li-OCOdO) + radical(OC=OOJ)""").
The thermo for this species is probably wrong! Setting CpInf = Cphigh for Entropy calculationat T = 2000.0 K...
ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.8846,97.4337,105.393],'J/(mol*K)'), H298=(-477.191,'kJ/mol'), S298=(269.551,'J/(mol*K)'), Cp0=(33.2579,'J/(mol*K)'), CpInf=(103.931,'J/(mol*K)'), comment="""Thermo group additivity estimation: group(O2s-(Cds-Cd)(Cds-Cd)) + group(O2s-(Cds-O2d)H) + group(Cds-OdOsOs) + group(Li-OCOdO) + radical(OC=OOJ)""").
The thermo for this species is probably wrong! Setting CpInf = Cphigh for En

In [26]:
def PDEP_possible(pdep_reaction):
    for family in database.kinetics.families:
        try:
            database.kinetics.families[family].add_atom_labels_for_reaction(pdep_reaction)
            template_labels = database.kinetics.families[family].get_reaction_template_labels(pdep_reaction)
            template = database.kinetics.families[family].retrieve_template(template_labels)
            kinetics = database.kinetics.families[family].get_kinetics_for_template(template, degeneracy=pdep_reaction.degeneracy)[0]
            pdep_reaction.kinetics = kinetics
            return family
        except (rmgpy.exceptions.UndeterminableKineticsError, rmgpy.exceptions.KineticsError, rmgpy.exceptions.ActionError, IndexError, ValueError):
            continue
    return None


# Make the official summary CSV

In [27]:
# Make a summary CSV

cols = ['rank', 'db_index', 'reaction', 'family', 'possible', 'avg_IS_pct_possible', 'uncertainty', 'avg_sens']
mech_summary = pd.DataFrame(columns=cols)

# improvement rank
# family is species, PDEP, or the reaction family
# possible is true (1) if we can calculate it, False otherwise
# avg_IS_pct_possible is the percent of the total possible improvement score this parameter represents


total_possible = 0
for i in range(len(avg_improvement_score)):
    if avg_improvement_score[i] > 0:
        if i < N:  # assume all species are possible
            total_possible += avg_improvement_score[i]
            continue
        
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[i - N]].family
        except AttributeError:
            pass
        # only these families are possible for reactions
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration', 'R_Addition_MultipleBond'] or PDEP_possible(reaction_list[ct2rmg_rxn[i - N]]):
            total_possible += avg_improvement_score[i]


# rank the parameters

parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


for i in range(0, 200):
    ct_index = improvement_order[i]
    
    if ct_index < N:
        # it's a species
        db_index = database_fun.get_unique_species_index(species_list[ct_index])
        mech_summary.loc[i] = [
            i,
            db_index,
            str(database_fun.index2species(db_index)),
            'species',
            1,
            np.round(avg_improvement_score[ct_index] / total_possible, 9),
            total_uncertainty_array[ct_index],
            avg_first_derivative[ct_index]
        ]   
    else:
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[ct_index - N]].family
        except AttributeError:
            pass
        
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
        improvement_percent = 0
        
        possible = 0
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration', 'R_Addition_MultipleBond'] or PDEP_possible(reaction_list[ct2rmg_rxn[ct_index - N]]):
            possible = 1
            improvement_percent = np.round(avg_improvement_score[ct_index] / total_possible, 9)

        mech_summary.loc[i] = [
            i,
            db_index,
            str(database_fun.index2reaction(db_index)),
            family,
            possible,
            improvement_percent,
            total_uncertainty_array[ct_index],
            avg_first_derivative[ct_index]
        ] 
        


In [28]:
mech_summary.to_csv(os.path.join(working_dir, 'mech_summary.csv'))